# BÀI TẬP: MEDICAL INSURANCE COST
**Nguồn:** kaggle.com/datasets/mosapabdelghany/medical-insurance-cost-dataset



## Setup

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')


csv_path = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv'

df = pd.read_csv(csv_path)
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [2]:
# TODO

display(df.shape)
display(df.columns.tolist())
display(df.dtypes)
df.info()

(1338, 7)

['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']

age           int64
sex          object
bmi         float64
children      int64
smoker       object
region       object
charges     float64
dtype: object

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


## A.2. Missing values & Duplicate data

In [3]:
# TODO

display(df.isnull().sum())
display(df.duplicated().sum())

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

np.int64(1)

## A.3. Invalid values

In [4]:
# TODO

display(((df['age'] < 0) | (df['age'] > 120)).sum())
display((df['bmi'] <= 0).sum())
display((df['charges'] <= 0).sum())
display((df['children'] < 0).sum())

np.int64(0)

np.int64(0)

np.int64(0)

np.int64(0)

## A.4. Create a new column
Tạo cột `bmi_group`: Normal (<25), Overweight (25-30), Obese (>=30).

In [5]:
# TODO
# A.4. Create a new column
df['bmi_group'] = np.where(df['bmi'] < 25, 'Normal', np.where(df['bmi'] < 30, 'Overweight', 'Obese'))

display(df['bmi_group'].value_counts())
display(df.head())

bmi_group
Obese         707
Overweight    386
Normal        245
Name: count, dtype: int64

,age,sex,bmi,children,smoker,region,charges,bmi_group
0,19,female,27.900,0,yes,southwest,16884.92400,Overweight
1,18,male,33.770,1,no,southeast,1725.55230,Obese
2,28,male,33.000,3,no,southeast,4449.46200,Obese
3,33,male,22.705,0,no,northwest,21984.47061,Normal
4,32,male,28.880,0,no,northwest,3866.85520,Overweight


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [6]:
# TODO

display(df[['age', 'bmi', 'children', 'charges']].mean())
display(df[['age', 'bmi', 'children', 'charges']].median())
display(df[['age', 'bmi', 'children', 'charges']].mode().iloc[0])

age            39.207025
bmi            30.663397
children        1.094918
charges     13270.422265
dtype: float64

age           39.000
bmi           30.400
children       1.000
charges     9382.033
dtype: float64

age           18.0000
bmi           32.3000
children       0.0000
charges     1639.5631
Name: 0, dtype: float64

## Group 2 — Dispersion

In [7]:
# TODO

stats_disp = pd.DataFrame({
    'Range': df[['age', 'bmi', 'charges']].max() - df[['age', 'bmi', 'charges']].min(),
    'Std': df[['age', 'bmi', 'charges']].std(),
    'IQR': df[['age', 'bmi', 'charges']].quantile(0.75) - df[['age', 'bmi', 'charges']].quantile(0.25),
    'CV': df[['age', 'bmi', 'charges']].std() / df[['age', 'bmi', 'charges']].mean()
})
display(stats_disp)

,Range,Std,IQR,CV
age,46.00000,14.049960,24.000000,0.358353
bmi,37.17000,6.098187,8.397500,0.198875
charges,62648.55411,12110.011237,11899.625365,0.912557


## Group 3 — Location and Shape

In [8]:
# TODO

stats_shape = pd.DataFrame({
    'P25': df[['bmi', 'charges']].quantile(0.25),
    'P50': df[['bmi', 'charges']].quantile(0.50),
    'P75': df[['bmi', 'charges']].quantile(0.75),
    'Skewness': stats.skew(df[['bmi', 'charges']]),
    'Kurtosis': stats.kurtosis(df[['bmi', 'charges']])
})
display(stats_shape)

,P25,P50,P75,Skewness,Kurtosis
bmi,26.29625,30.400,34.693750,0.283729,-0.055023
charges,4740.28715,9382.033,16639.912515,1.514180,1.595821


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Người hút thuốc trả chi phí cao hơn bao nhiêu lần so với người không hút, và có đồng đều giữa các vùng không?

In [9]:
# TODO

charges_smoker = df.groupby('smoker')['charges'].mean()
ratio_smoker = charges_smoker['yes'] / charges_smoker['no']
region_smoker = df.pivot_table(index='region', columns='smoker', values='charges', aggfunc='mean')
region_smoker['ratio'] = region_smoker['yes'] / region_smoker['no']
display(region_smoker)
print(f"=> Người hút thuốc trả chi phí trung bình {charges_smoker['yes']:,.2f}, cao gấp {ratio_smoker:.2f} lần người không hút ({charges_smoker['no']:,.2f}).")
print(f"=> Tỷ lệ chi phí chênh lệch theo từng vùng dao động từ {region_smoker['ratio'].min():.2f} đến {region_smoker['ratio'].max():.2f} lần .")

smoker,no,yes,ratio
region,,,
northeast,9165.531672,29673.536473,3.237514
northwest,8556.463715,30192.003182,3.528561
southeast,8032.216309,34844.996824,4.338155
southwest,8019.284513,32269.063494,4.023933


=> Người hút thuốc trả chi phí trung bình 32,050.23, cao gấp 3.80 lần người không hút (8,434.27).
=> Tỷ lệ chi phí chênh lệch theo từng vùng dao động từ 3.24 đến 4.34 lần .


## Câu hỏi 2: BMI có tương quan với chi phí mạnh hơn ở nhóm hút thuốc hay không hút thuốc?

In [10]:
# TODO

corr_yes = df[df['smoker'] == 'yes']['bmi'].corr(df[df['smoker'] == 'yes']['charges'])
corr_no = df[df['smoker'] == 'no']['bmi'].corr(df[df['smoker'] == 'no']['charges'])
print(f"Tương quan BMI vs Charges (Hút thuốc): {corr_yes:.3f}")
print(f"Tương quan BMI vs Charges (Không hút thuốc): {corr_no:.3f}")


Tương quan BMI vs Charges (Hút thuốc): 0.806
Tương quan BMI vs Charges (Không hút thuốc): 0.084


## Câu hỏi 3: Vùng nào có chi phí bảo hiểm trung bình cao nhất?

In [11]:
# TODO

charges_by_region = df.groupby('region')['charges'].mean().sort_values(ascending=False)
display(charges_by_region.round(2))
print(f"\n=> Vùng có chi phí bảo hiểm trung bình cao nhất là {charges_by_region.index[0]} ({charges_by_region.iloc[0]:,.2f}).")

region
southeast    14735.41
northeast    13406.38
northwest    12417.58
southwest    12346.94
Name: charges, dtype: float64


=> Vùng có chi phí bảo hiểm trung bình cao nhất là southeast (14,735.41).


## Câu hỏi 4: Số lượng con cái có làm tăng chi phí bảo hiểm không?

In [12]:
# TODO

display(df.groupby('children')['charges'].mean().round(2))
corr_children = df['children'].corr(df['charges'])
print(f"\nTương quan giữa số con và chi phí: {corr_children:.3f}")
print("=> Số lượng con cái không làm tăng chi phí bảo hiểm đáng kể.")

children
0    12365.98
1    12731.17
2    15073.56
3    15355.32
4    13850.66
5     8786.04
Name: charges, dtype: float64


Tương quan giữa số con và chi phí: 0.068
=> Số lượng con cái không làm tăng chi phí bảo hiểm đáng kể.


## Câu hỏi 5: Tuổi có tương quan với chi phí bảo hiểm không?

In [13]:
# TODO

corr_age = df['age'].corr(df['charges'])
print(f"Tương quan giữa Tuổi và Chi phí: {corr_age:.3f}")


Tương quan giữa Tuổi và Chi phí: 0.299


## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

*(Viết insight của bạn vào đây...)*
Người hút thuốc chịu chi phí bảo hiểm cao gấp gần 4 lần người không hút thuốc.
Tuổi càng cao thì chi phí viện phí càng tăng đều.  Vùng Southeast có chi phí trung bình cao nhất.